# Домашнее Задание

### Моделирование
Обучить модель линейной регресии на всех фичах (категориальных+числовых).
Предварительно преобразовать категориальные фичи, заполнить пропуски, почистить высокоскореллированные фичи и тд.

Попробовать обучить lasso/ridge регрессии, оценить коэффициенты при регрессорах, описать есть ли ненужные фичи.

Замерить метрики MSE, R2 на train/test выборках

Со **здвёздочкой***: Дополнительно отобрать фичи таким образом, чтобы максимизировать качество модели на test выборке.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
import sklearn.preprocessing as ppc
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso

In [2]:
df = pd.read_csv("../../assets/house_prices.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [3]:
target_col = "SalePrice"
df.drop(columns=["Id"], inplace=True)

In [4]:
(X_train, X_test, y_train, y_test) = train_test_split(
    df.drop(columns=[target_col], axis=1),
    df[target_col],
    test_size=0.3,
    random_state=69,
)

In [5]:
def find_high_corr_cols(df: pd.DataFrame, threshold: float = 0.75) -> pd.DataFrame:
    corr_matrix = df.corr(numeric_only=True)
    high_corr_pairs = []

    for i in range(len(corr_matrix.columns)):
        for j in range(i + 1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) >= threshold:
                high_corr_pairs.append(
                    (
                        corr_matrix.columns[i],
                        corr_matrix.columns[j],
                        corr_matrix.iloc[i, j],
                    )
                )

    high_corr_df = pd.DataFrame(
        high_corr_pairs,
        columns=[
            "Feature 1",
            "Feature 2",
            "Correlation",
        ],
    )
    high_corr_df = high_corr_df.sort_values("Correlation", ascending=False)

    return high_corr_df

In [6]:
df_high_corr = find_high_corr_cols(X_train)

print(df_high_corr)

X_train = X_train.drop(columns=df_high_corr["Feature 2"].to_list(), axis=1)
X_test = X_test.drop(columns=df_high_corr["Feature 2"].to_list(), axis=1)

     Feature 1     Feature 2  Correlation
3   GarageCars    GarageArea     0.883277
2    GrLivArea  TotRmsAbvGrd     0.835103
0    YearBuilt   GarageYrBlt     0.821062
1  TotalBsmtSF      1stFlrSF     0.820656


In [7]:
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("category features: ", categorical_cols)

category features:  ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']


In [8]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print("numerical features: ", num_cols)

numerical features:  ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'Fireplaces', 'GarageCars', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']


In [9]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", ppc.StandardScaler()),
        (
            "transformer",
            ppc.SplineTransformer(
                n_knots=7,
                degree=7,
                order="F",
                extrapolation="linear",
            ),
        ),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)


pipe_ridge = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("poly", ppc.PolynomialFeatures(2)),
        (
            "regressor",
            Ridge(
                alpha=15,
                max_iter=None,
                random_state=69,
            ),
        ),
    ]
)

pipe_lasso = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("poly", ppc.PolynomialFeatures(2)),
        (
            "regressor",
            Lasso(
                alpha=0.001,
                random_state=69,
                max_iter=1000,
            ),
        ),
    ]
)

In [10]:
clf_ridge = TransformedTargetRegressor(
    regressor=pipe_ridge,
    func=np.log1p,
    inverse_func=np.expm1,
)

clf_lasso = TransformedTargetRegressor(
    regressor=pipe_lasso,
    func=np.log1p,
    inverse_func=np.expm1,
)

In [11]:
clf_ridge.fit(X_test, y_test)

y_pred = clf_ridge.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)

print("test:\n")
print("r2:\t", r2)
print("rmse:\t", rmse)

test:

r2:	 0.9988035998755035
rmse:	 6846245.840946668


In [12]:
clf_ridge.fit(X_train, y_train)

y_pred = clf_ridge.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)

print("train:\n")
print("r2:\t", r2)
print("rmse:\t", rmse)

train:

r2:	 0.8799932324015712
rmse:	 686723293.2644007


In [13]:
clf_lasso.fit(X_test, y_test)

y_pred = clf_lasso.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)

print("test:\n")
print("r2:\t", r2)
print("rmse:\t", rmse)

test:

r2:	 0.9679063946150622
rmse:	 183651529.1906013


In [14]:
clf_lasso.fit(X_train, y_train)

y_pred = clf_lasso.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)

print("train:\n")
print("r2:\t", r2)
print("rmse:\t", rmse)

c:\demiskira\projects\fefu\sem6\ds\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.368e-02, tolerance: 1.674e-02
  model = cd_fast.enet_coordinate_descent(


train:

r2:	 0.8759888614979869
rmse:	 709637790.74977


In [15]:
fitted_preprocessor = clf_ridge.regressor_.named_steps["preprocessor"]
preprocessor_feature_names = fitted_preprocessor.get_feature_names_out()

poly = clf_ridge.regressor_.named_steps["poly"]

X_preprocessed = fitted_preprocessor.transform(X_train.iloc[:1])
poly.fit(X_preprocessed)

feature_names = poly.get_feature_names_out(preprocessor_feature_names)

ridge_coefs = clf_ridge.regressor_.named_steps["regressor"].coef_
ridge_importance = pd.DataFrame({"Feature": feature_names, "Coefficient": ridge_coefs})
ridge_importance = ridge_importance.sort_values("Coefficient", key=abs, ascending=False)
print("Top 10 most important features (Ridge):")
print(ridge_importance.head(10))

Top 10 most important features (Ridge):
                                                  Feature  Coefficient
199506            cat__LotShape_Reg cat__GarageFinish_RFn    -0.010365
199473                cat__LotShape_Reg cat__HeatingQC_TA    -0.010261
228744  cat__GarageType_Attchd cat__SaleCondition_Abnorml     0.009289
204255    cat__Neighborhood_CollgCr cat__GarageFinish_Fin    -0.009163
221211            cat__MasVnrType_nan cat__FireplaceQu_Gd     0.008684
199377               cat__LotShape_Reg cat__RoofStyle_Hip    -0.008625
198630               cat__LotShape_IR1 cat__RoofStyle_Hip     0.008435
204256    cat__Neighborhood_CollgCr cat__GarageFinish_RFn     0.008227
199375             cat__LotShape_Reg cat__RoofStyle_Gable     0.008136
199471                cat__LotShape_Reg cat__HeatingQC_Gd     0.008129


In [16]:
lasso_coefs = clf_lasso.regressor_.named_steps["regressor"].coef_
lasso_importance = pd.DataFrame({"Feature": feature_names, "Coefficient": lasso_coefs})
lasso_importance = lasso_importance.sort_values("Coefficient", key=abs, ascending=False)
print("\nTop 10 most important features (Lasso):")
print(lasso_importance.head(10))

zero_coefs = (lasso_coefs == 0).sum()
print(f"\nLasso обнулил {zero_coefs} из {len(lasso_coefs)} коэффициентов")


Top 10 most important features (Lasso):
                                            Feature  Coefficient
109369          num__GrLivArea_sp_3 cat__PoolQC_nan    -0.443274
89293   num__TotalBsmtSF_sp_3 cat__RoofMatl_CompShg    -0.203308
32423          num__OverallQual_sp_9 cat__Alley_nan     0.181870
110610         num__GrLivArea_sp_6 cat__Street_Pave     0.166992
186                             num__GrLivArea_sp_3    -0.145444
31803   num__OverallQual_sp_8 cat__Utilities_AllPub     0.138288
156836      num__GarageCars_sp_8 cat__BldgType_1Fam     0.131372
90463     num__TotalBsmtSF_sp_5 cat__Functional_Typ     0.120170
44                            num__OverallQual_sp_4    -0.119515
130438       num__FullBath_sp_9 cat__Functional_Typ     0.112539

Lasso обнулил 229919 из 230181 коэффициентов
